In [ ]:
import os
import datetime

# 1. 분석 범위 설정
start_date = datetime.date(2016, 1, 1)
end_date = datetime.date(2023, 12, 31)

# 2. 날짜 리스트 생성
expected_dates = set((start_date + datetime.timedelta(days=i)).strftime('%Y.%m.%d')
                     for i in range((end_date - start_date).days + 1))

# 3. 폴더 경로 및 파일 리스트
folder_path = r'G:\24.11.Graduate\1.CoKriging'
file_list = os.listdir(folder_path)

# 4. 파일 이름에서 날짜 추출
available_dates = set()
for filename in file_list:
    if filename.endswith('.nc') and '_Cokriging_Precipitation' in filename:
        date_part = filename.split('_')[0]
        try:
            datetime.datetime.strptime(date_part, '%Y.%m.%d')  # 유효한 날짜인지 확인
            available_dates.add(date_part)
        except ValueError:
            continue  # 형식이 잘못된 경우 무시

# 5. 누락된 날짜 계산
missing_dates = sorted(expected_dates - available_dates)

# 6. 결과 출력
print(f"총 누락된 날짜 수: {len(missing_dates)}")
if missing_dates:
    print("누락된 날짜 예시 (최대 30개):")
    for date in missing_dates[:30]:
        print(date)
else:
    print("모든 날짜의 파일이 존재합니다.")

In [ ]:
import os
import xarray as xr
import datetime
import pandas as pd

# 1) 기본 경로 설정
input_folder = r'G:\24.11.Graduate\1.CoKriging'
base_output  = r'C:\Yeonji\2025.01.Drought'

# 2) 날짜별 파일 매핑
file_list = sorted([
    f for f in os.listdir(input_folder)
    if f.endswith('.nc') and '_Cokriging_Precipitation' in f
])
date_file_map = {}
for fname in file_list:
    date_str = fname.split('_')[0]  # 'YYYY.MM.DD'
    try:
        dt = datetime.datetime.strptime(date_str, "%Y.%m.%d")
        # pandas Timestamp 로 변환해야 그룹핑 편리
        ts = pd.Timestamp(dt.date())
        date_file_map[ts] = os.path.join(input_folder, fname)
    except Exception as e:
        print(f"❌ 날짜 파싱 실패: {fname} → {e}")

# 3) 월별 날짜 그룹핑
all_dates = sorted(date_file_map.keys())
df = pd.DataFrame({'date': all_dates})
df['ym'] = df['date'].dt.to_period('M').astype(str)  # 'YYYY-MM'
monthly_dates = df.groupby('ym')['date'].apply(list).to_dict()
sorted_months = sorted(monthly_dates.keys())

# 4) 누적 기간 정의
window_sizes = [3, 6, 9, 12]

for window in window_sizes:
    out_folder = os.path.join(base_output, f'{window}mon_output_Cokriging')
    os.makedirs(out_folder, exist_ok=True)

    # 슬라이딩 윈도우
    for i in range(len(sorted_months) - window + 1):
        months = sorted_months[i:i+window]
        dates = sum((monthly_dates[m] for m in months), [])
        out_yyyymm = months[-1].replace('-', '')  # e.g. '202503'

        ds_list = []
        for date in dates:
            path = date_file_map.get(date)
            if not path or not os.path.exists(path):
                print(f"⚠️ 파일 없음: {date} → {path}")
                continue
            ds = xr.open_dataset(path)
            if 'precipitation' not in ds:
                print(f"⚠️ 변수 없음: {path}")
                ds.close()
                continue
            ds_list.append(ds)

        # 데이터를 충분히 모았는지 체크 (window*20일 정도 기준)
        if len(ds_list) < window * 20:
            print(f"⚠️ 스킵 {out_yyyymm} ({window}개월, 일수 부족: {len(ds_list)})")
            for ds in ds_list: ds.close()
            continue

        # 누적 합 계산
        total = sum(ds['precipitation'] for ds in ds_list)
        result = ds_list[0].copy(deep=True)
        result['precipitation'] = total

        # 저장
        out_path = os.path.join(
            out_folder,
            f"{out_yyyymm}_Cokriging_{window}mon_Precipitation.nc"
        )
        result.to_netcdf(out_path)
        print(f"✅ 저장 ({window}개월): {out_path}")

        # 리소스 해제
        for ds in ds_list: ds.close()
        result.close()


In [ ]:
import os
import xarray as xr
import numpy as np
import scipy.stats as stats
import pandas as pd
from tqdm import tqdm

# 기본 경로
base_folder = r'C:\Yeonji\2025.01.Drought'

# 사용할 누적 개월 수
window_sizes = [6, 9, 12]

def compute_spi_series(values):
    """1D 시계열(values)에 대해 감마분포 기반 SPI 계산"""
    arr = np.array(values)
    if np.all(np.isnan(arr)) or np.sum(arr > 0) < 30:
        return np.full_like(arr, np.nan)
    try:
        a, loc, scale = stats.gamma.fit(arr, floc=0)
        cdf = stats.gamma.cdf(arr, a, loc=loc, scale=scale)
        return stats.norm.ppf(cdf)
    except Exception:
        return np.full_like(arr, np.nan)

for window in window_sizes:
    # 입력·출력 폴더 설정
    in_folder  = os.path.join(base_folder, f'{window}mon_output_Cokriging')
    out_folder = os.path.join(base_folder, f'SPI{window}(Cok)')
    os.makedirs(out_folder, exist_ok=True)

    # 파일 모으기
    file_list = sorted([
        f for f in os.listdir(in_folder)
        if f.endswith('.nc') and f'{window}mon' in f
    ])
    dates = [pd.to_datetime(f.split('_')[0], format='%Y%m') for f in file_list]

    # xarray Dataset 리스트 생성
    datasets = []
    for fname, date in zip(file_list, dates):
        ds = xr.open_dataset(os.path.join(in_folder, fname))
        ds = ds.expand_dims(time=[np.datetime64(date, 'ns')])
        datasets.append(ds)

    # 병합
    full = xr.concat(datasets, dim='time')
    precip = full['precipitation']  # (time, lat, lon)

    # SPI 계산을 위한 배열 초기화
    nt, ny, nx = precip.shape
    spi_arr = np.full((nt, ny, nx), np.nan)

    for i in tqdm(range(ny), desc=f'SPI{window} 계산 (위도)'):
        for j in range(nx):
            spi_arr[:, i, j] = compute_spi_series(precip[:, i, j].values)

    # xarray Dataset으로 재구성
    spi_ds = xr.Dataset(
        {f'SPI{window}': (['time', 'lat', 'lon'], spi_arr)},
        coords={
            'time': full.time,
            'lat': full.lat,
            'lon': full.lon
        }
    )

    # 시간별로 분할 저장
    for idx, ts in enumerate(spi_ds.time.values):
        ym = pd.to_datetime(str(ts)).strftime('%Y%m')
        single = (
            spi_ds.isel(time=idx)
                  .drop_vars('time')
                  .expand_dims(time=[ts])
        )
        out_file = os.path.join(out_folder, f"{ym}_SPI{window}.nc")
        single.to_netcdf(out_file)
        print(f"✅ 저장됨: {out_file}")

    # 메모리 해제
    full.close()
    spi_ds.close()
    for ds in datasets:
        ds.close()


In [ ]:
import os
import xarray as xr
import datetime
import pandas as pd

# 1) 경로 설정
input_folder  = r'C:\Yeonji\2025.01.Drought\GAN_Daily'
output_folder = r'C:\Yeonji\2025.01.Drought\3mon_output(GAN)'
os.makedirs(output_folder, exist_ok=True)

# 2) 파일 리스트
file_list = sorted([
    f for f in os.listdir(input_folder)
    if f.endswith('.nc') and '_GAN_Prediction' in f
])
print("찾은 파일:", file_list[:5], "...")  # 확인용

# 3) 날짜 매핑: Python date 대신 pandas Timestamp 사용
date_file_map = {}
for fname in file_list:
    date_str = fname.split('_')[0]  # 'YYYYMMDD'
    try:
        # Timestamp 로 변환 (시간 정보 없이 날짜만)
        ts = pd.to_datetime(date_str, format='%Y%m%d')
        date_file_map[ts] = os.path.join(input_folder, fname)
    except ValueError:
        print(f"❌ 날짜 파싱 실패: {fname}")
        continue

# 4) DataFrame 생성 및 월별 그룹핑
all_dates = sorted(date_file_map.keys())      # 이제 Timestamp 리스트
df = pd.DataFrame({'date': all_dates})
df['ym'] = df['date'].dt.to_period('M').astype(str)
monthly_dates = df.groupby('ym')['date'].apply(list).to_dict()
sorted_months = sorted(monthly_dates)

# 5) 3개월 이동 누적 계산
for i in range(len(sorted_months) - 2):
    m1, m2, m3 = sorted_months[i:i+3]
    dates = monthly_dates[m1] + monthly_dates[m2] + monthly_dates[m3]
    out_ym = m3.replace('-', '')  # 'YYYYMM'

    ds_list = []
    for ts in dates:
        path = date_file_map.get(ts)  # pandas Timestamp 그대로 조회
        if not path or not os.path.exists(path):
            print(f"⚠️ 파일 없음: {ts} -> {path}")
            continue
        ds = xr.open_dataset(path)
        if 'precipitation' not in ds:
            print(f"⚠️ 변수 없음: {path}")
            ds.close()
            continue
        ds_list.append(ds)

    # 데이터 개수 체크 (약 90일)
    if len(ds_list) < 80:
        print(f"⚠️ 스킵 {out_ym} (일수 부족: {len(ds_list)})")
        for ds in ds_list: ds.close()
        continue

    # 누적
    total = sum(ds['precipitation'] for ds in ds_list)
    result = ds_list[0].copy(deep=True)
    result['precipitation'] = total

    # 저장
    out_path = os.path.join(output_folder, f"{out_ym}_GAN_3mon_Precipitation.nc")
    result.to_netcdf(out_path)
    print("✅ 저장:", out_path)

    # 리소스 정리
    for ds in ds_list:
        ds.close()
    result.close()


In [ ]:
import os
import xarray as xr
import pandas as pd

# 1) 기본 경로 설정
input_folder  = r'C:\Yeonji\2025.01.Drought\GAN_Daily'
output_base   = r'C:\Yeonji\2025.01.Drought'

# 2) 파일 리스트 및 날짜 매핑
file_list = sorted(f for f in os.listdir(input_folder)
                   if f.endswith('.nc') and '_GAN_Prediction' in f)
date_file_map = {}
for fname in file_list:
    date_str = fname.split('_')[0]  # 'YYYYMMDD'
    try:
        ts = pd.to_datetime(date_str, format='%Y%m%d')
        date_file_map[ts] = os.path.join(input_folder, fname)
    except ValueError:
        print(f"❌ 날짜 파싱 실패: {fname}")

# 3) 월별 날짜 그룹핑
all_dates = sorted(date_file_map.keys())
df = pd.DataFrame({'date': all_dates})
df['ym'] = df['date'].dt.to_period('M').astype(str)
monthly_dates = df.groupby('ym')['date'].apply(list).to_dict()
sorted_months = sorted(monthly_dates)

# 4) 누적 기간 리스트
window_sizes = [6, 9, 12]  # 원하는 누적 개월 수

for window in window_sizes:
    # 기간별 출력 폴더
    out_folder = os.path.join(output_base, f"{window}mon_output(GAN)")
    os.makedirs(out_folder, exist_ok=True)

    # sliding window
    for i in range(len(sorted_months) - window + 1):
        months = sorted_months[i:i+window]
        # 해당 기간의 모든 날짜 목록
        dates = sum((monthly_dates[m] for m in months), [])
        out_ym = months[-1].replace('-', '')  # YYYYMM of 마지막 달
        
        ds_list = []
        for ts in dates:
            path = date_file_map.get(ts)
            if not path or not os.path.exists(path):
                print(f"⚠️ 파일 없음: {ts} -> {path}")
                continue
            ds = xr.open_dataset(path)
            if 'precipitation' not in ds:
                print(f"⚠️ 변수 없음: {path}")
                ds.close()
                continue
            ds_list.append(ds)
        
        # 충분한 일수(대략 window*25일) 검증
        if len(ds_list) < window * 25:
            print(f"⚠️ 스킵 {out_ym} ({window}개월, 일수 부족: {len(ds_list)})")
            for ds in ds_list: ds.close()
            continue
        
        # 누적 합 계산
        total = sum(ds['precipitation'] for ds in ds_list)
        result = ds_list[0].copy(deep=True)
        result['precipitation'] = total
        
        # 저장
        out_path = os.path.join(
            out_folder,
            f"{out_ym}_GAN_{window}mon_Precipitation.nc"
        )
        result.to_netcdf(out_path)
        print(f"✅ 저장 ({window}개월): {out_path}")
        
        # 리소스 정리
        for ds in ds_list: ds.close()
        result.close()


In [ ]:
import os
import xarray as xr
import numpy as np
import scipy.stats as stats
import pandas as pd
from tqdm import tqdm

# 기본 경로 설정
base_folder = r'C:\Yeonji\2025.01.Drought'
window_sizes = [6, 9, 12]  # 원하는 누적 개월 수

def compute_spi_series(values):
    """1D 시계열(values)에 대해 감마분포 기반 SPI 계산"""
    arr = np.array(values)
    if np.all(np.isnan(arr)) or np.sum(arr > 0) < 30:
        return np.full_like(arr, np.nan)
    try:
        a, loc, scale = stats.gamma.fit(arr, floc=0)
        cdf = stats.gamma.cdf(arr, a, loc=loc, scale=scale)
        return stats.norm.ppf(cdf)
    except Exception:
        return np.full_like(arr, np.nan)

for window in window_sizes:
    input_folder  = os.path.join(base_folder, f'{window}mon_output(GAN)')
    output_folder = os.path.join(base_folder, f'SPI{window}(GAN)')
    os.makedirs(output_folder, exist_ok=True)

    # 윈도우별 파일 수집 (예: "202501_GAN_6mon_Precipitation.nc")
    file_list = sorted([
        f for f in os.listdir(input_folder)
        if f.endswith('.nc') and f'_{window}mon_' in f
    ])
    if not file_list:
        print(f"⚠️ 입력 폴더에 파일이 없습니다: {input_folder}")
        continue

    # 파일명에서 YYYYMM 추출
    dates = [pd.to_datetime(f.split('_')[0], format='%Y%m') for f in file_list]

    # xarray Dataset 리스트로 불러와서 시간 축 추가
    datasets = []
    for fname, date in zip(file_list, dates):
        ds = xr.open_dataset(os.path.join(input_folder, fname))
        ds = ds.expand_dims(time=[np.datetime64(date, 'ns')])
        datasets.append(ds)

    # 하나의 Dataset으로 병합
    full = xr.concat(datasets, dim='time')
    precip = full['precipitation']  # shape: (time, lat, lon)

    # 메모리 절약을 위해 원본 Dataset 닫기
    for ds in datasets:
        ds.close()

    # SPI 계산
    time_len, lat_len, lon_len = precip.shape
    spi_arr = np.full((time_len, lat_len, lon_len), np.nan)

    for i in tqdm(range(lat_len), desc=f'SPI{window} 계산 (위도)'):
        for j in range(lon_len):
            spi_arr[:, i, j] = compute_spi_series(precip[:, i, j].values)

    # xarray Dataset으로 재구성
    spi_ds = xr.Dataset(
        {f'SPI{window}': (['time', 'latitude', 'longitude'], spi_arr)},
        coords={
            'time': full.time,
            'latitude': full.latitude,
            'longitude': full.longitude
        }
    )

    # 시간별로 분할 저장
    for idx, ts in enumerate(spi_ds.time.values):
        ym = pd.to_datetime(str(ts)).strftime('%Y%m')
        slice_ds = (
            spi_ds.isel(time=idx)
                  .drop_vars('time')
                  .expand_dims(time=[ts])
        )
        out_file = os.path.join(output_folder, f"{ym}_SPI{window}.nc")
        slice_ds.to_netcdf(out_file)
        print(f"✅ 저장됨: {out_file}")

    # 마무리
    full.close()
    spi_ds.close()
